# TensorRT

Di notebook ini, kita akan menggunakan TensorRT untuk mengoptimalkan model PyTorch untuk inferensi. Kami akan melatih model CNN sederhana pada kumpulan data MNIST, mengonversinya menjadi mesin TensorRT menggunakan ONNX, lalu melakukan inferensi menggunakan model mesin TensorRT yang dioptimalkan dan mengevaluasi ukuran dan keakuratan model. Notebook ini memerlukan GPU NVIDIA dengan dukungan CUDA atau perangkat NVIDIA Jetson.

## Siapkan TensorRT

Pertama, instal tensorrt dan torch menggunakan pip dan impor modul yang diperlukan

In [1]:
%pip install torch torchvision
%pip install tensorrt==8.6.1
%pip install pycuda onnx onnxruntime
%pip install --no-cache-dir --extra-index-url https://pypi.nvidia.com pytorch-quantization==2.1.2

  Preparing metadata (setup.py) ... done
  Created wheel for tensorrt: filename=tensorrt-8.6.1-py2.py3-none-any.whl size=16972 sha256=cc31e3c094d7965b747075a7e397e004d74cbf5b6e78370d04f2ce3f6d7577d8
  Stored in directory: /root/.cache/pip/wheels/6d/29/56/abdffd4c604f255b5254bef3f1c598ab7811ea020540599438
Successfully built tensorrt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.4/92.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.

In [2]:
import torch  # Library utama untuk operasi tensor dan jaringan neural menggunakan PyTorch
import torch.nn as nn  # Modul untuk mendefinisikan lapisan-lapisan jaringan neural dalam PyTorch
import torch.nn.functional as F  # Modul untuk fungsi-fungsi aktivasi dan operasi lainnya
import torch.optim as optim  # Modul untuk berbagai algoritma optimisasi seperti Adam atau SGD
from torchvision import datasets, transforms  # Mengimpor dataset standar dan transformasi dari torchvision
import torch.quantization  # Modul untuk mengonversi model ke bentuk quantized dengan PyTorch
import pathlib  # Modul untuk manipulasi path file secara lintas platform
import numpy as np  # Library untuk operasi numerik dengan array multidimensi
import torch.onnx  # Modul untuk mengekspor model PyTorch ke format ONNX
import tensorrt as trt  # Library untuk melakukan inferensi menggunakan TensorRT, optimasi untuk GPU
import pycuda.driver as cuda  # Library untuk menggunakan CUDA (Compute Unified Device Architecture) dengan PyCUDA
import pycuda.autoinit  # Menginisialisasi PyCUDA secara otomatis untuk menggunakan GPU
import onnx  # Library untuk memanipulasi dan bekerja dengan model ONNX
import onnxruntime  # Mesin inferensi untuk menjalankan model ONNX di berbagai perangkat
from pytorch_quantization import nn as quant_nn  # Mengimpor modul quantization dari pytorch_quantization untuk PyTorch
from pytorch_quantization import quant_modules  # Menyediakan berbagai modul quantization untuk PyTorch
from pytorch_quantization import calib  # Menyediakan kalibrasi untuk quantization pada model PyTorch
from tqdm import tqdm  # Mengimpor modul untuk progress bar yang lebih mudah dibaca


## Latih Model PyTorch dan Ekspor ke ONNX

Selanjutnya, latih model CNN sederhana pada kumpulan data MNIST dan ekspor ke format ONNX

In [3]:
# Transformasi untuk mempersiapkan data (mengubah gambar menjadi tensor dan normalisasi)
transform = transforms.Compose([
    transforms.ToTensor(),  # Mengubah gambar menjadi tensor PyTorch
    transforms.Normalize((0.1307,), (0.3081,))  # Normalisasi gambar berdasarkan mean dan std dataset MNIST
])

# Memuat dataset MNIST untuk pelatihan dan pengujian
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)  # Data pelatihan
test_dataset = datasets.MNIST('./data', train=False, transform=transform)  # Data pengujian

# Definisi arsitektur jaringan neural sederhana
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=12, kernel_size=3)  # Lapisan konvolusi
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Lapisan pooling
        self.fc = nn.Linear(12 * 13 * 13, 10)  # Lapisan fully connected untuk klasifikasi ke 10 kelas

    def forward(self, x):
        x = x.view(-1, 1, 28, 28)  # Mengubah ukuran gambar untuk lapisan konvolusi
        x = F.relu(self.conv1(x))  # Aktivasi ReLU setelah konvolusi
        x = self.pool(x)  # Pooling untuk mereduksi dimensi spasial
        x = x.view(x.size(0), -1)  # Flatten tensor sebelum masuk ke lapisan fully connected
        x = self.fc(x)  # Lapisan fully connected untuk klasifikasi
        output = F.log_softmax(x, dim=1)  # Log-softmax untuk output
        return output

# DataLoader untuk memuat data pelatihan dan pengujian dalam batch
train_loader = torch.utils.data.DataLoader(train_dataset, 32)
test_loader = torch.utils.data.DataLoader(test_dataset, 32)

# Menentukan perangkat untuk pelatihan (CPU dalam hal ini)
device = "cpu"

# Jumlah epoch untuk pelatihan
epochs = 1

# Membuat model dan memindahkannya ke perangkat (CPU)
model = Net().to(device)

# Menggunakan optimasi Adam untuk melatih model
optimizer = optim.Adam(model.parameters())

# Mengaktifkan mode pelatihan pada model
model.train()

# Loop pelatihan
for epoch in range(1, epochs+1):
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)  # Memindahkan data ke perangkat
        optimizer.zero_grad()  # Mengatur gradien ke nol
        output = model(data)  # Melakukan forward pass
        loss = F.nll_loss(output, target)  # Menghitung loss (Negative Log-Likelihood)
        loss.backward()  # Melakukan backpropagation untuk menghitung gradien
        optimizer.step()  # Memperbarui bobot model
        print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
            epoch, batch_idx * len(data), len(train_loader.dataset),
            100. * batch_idx / len(train_loader), loss.item()))  # Menampilkan progres pelatihan

# Menyimpan model yang telah dilatih ke direktori 'models'
MODEL_DIR = pathlib.Path("./models")
MODEL_DIR.mkdir(exist_ok=True)  # Membuat direktori jika belum ada
torch.save(model.state_dict(), MODEL_DIR / "original_model.p")  # Menyimpan bobot model

# Mengambil batch pertama dari train_loader untuk digunakan sebagai input
x, _ = next(iter(train_loader))

# Mengekspor model PyTorch ke format ONNX
torch.onnx.export(model,
                  x,  # Menggunakan batch pertama sebagai input
                  MODEL_DIR / "mnist_model.onnx",  # Lokasi untuk menyimpan model ONNX
                  export_params=True,  # Mengekspor bobot model
                  opset_version=10,  # Versi opset ONNX
                  do_constant_folding=True,  # Melakukan folding pada operasi konstan
                  input_names=['input'],  # Nama input tensor
                  output_names=['output'],  # Nama output tensor
                  dynamic_axes={'input' : {0 : 'batch_size'},  # Mendefinisikan dimensi dinamis untuk batch size
                                'output' : {0 : 'batch_size'}})  # Mendefinisikan dimensi dinamis untuk batch size


Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9.91M/9.91M [00:00<00:00, 17.5MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28.9k/28.9k [00:00<00:00, 514kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1.65M/1.65M [00:00<00:00, 3.84MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4.54k/4.54k [00:00<00:00, 10.3MB/s]


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.428713
Train Epoch: 1 [32/60000 (0%)]	Loss: 2.462280
Train Epoch: 1 [64/60000 (0%)]	Loss: 2.208610
Train Epoch: 1 [96/60000 (0%)]	Loss: 2.027727
Train Epoch: 1 [128/60000 (0%)]	Loss: 2.094260
Train Epoch: 1 [160/60000 (0%)]	Loss: 2.110198
Train Epoch: 1 [192/60000 (0%)]	Loss: 1.657317
Train Epoch: 1 [224/60000 (0%)]	Loss: 1.784680
Train Epoch: 1 [256/60000 (0%)]	Loss: 1.808799
Train Epoch: 1 [288/60000 (0%)]	Loss: 1.449356
Train Epoch: 1 [320/60000 (1%)]	Loss: 1.527941
Train Epoch: 1 [352/60000 (1%)]	Loss: 1.307230
Train Epoch: 1 [384/60000 (1%)]	Loss: 1.368755
Train Epoch: 1 [416/60000 (1%)]	Loss: 1.246383
Train Epoch: 1 [448/60000 (1%)]	Loss: 1.105210
Train Epoch: 1 [480/60000 (1%)]	Loss: 1.494388
Train Epoch: 1 [512/60000 (1%)]	Loss: 1.288670
Train Epoch: 1 [544/60000 (1%)]	Loss: 0.987826
Train Epoch: 1 [576/60000 (1%)]	Loss: 1.250500
Train Epoch: 1 [608/60000 (1%)]	Loss:

## Konversikan Model ONNX ke TensorRT

Untuk mengonversi model ONNX ke mesin TensorRT menggunakan TensorRT Python API. Pertama, inisialisasi komponen TensorRT yaitu logger, builder, dan jaringan. Selanjutnya, tentukan parser ONNX untuk mengurai model ONNX dari file ONNX ke jaringan TensorRT. Kemudian, buat konfigurasi pembangun untuk menetapkan parameter pembangunan dan batas kumpulan memori untuk ruang kerja di TensorRT. Kemudian, buat profil pengoptimalan untuk menangani bentuk masukan dinamis dengan ukuran batch 32, ukuran saluran 1, dan dimensi gambar 28x28. Selanjutnya, buat dan buat serial mesin TensorRT menggunakan jaringan dan pembuat yang dikonfigurasi, lalu simpan ke disk. Terakhir, skrip dibersihkan dengan menghapus pembuat dan objek jaringan untuk mengosongkan sumber daya.

In [4]:
# Menentukan path model ONNX dan file TensorRT yang akan disimpan
onnx_path = MODEL_DIR / "mnist_model.onnx"
trt_path = MODEL_DIR / 'mnist_engine_pytorch.trt'

# Inisialisasi logger dan builder untuk TensorRT
logger = trt.Logger(trt.Logger.WARNING)  # Membuat logger untuk TensorRT dengan level warning
builder = trt.Builder(logger)  # Membuat builder TensorRT menggunakan logger
# Membuat jaringan TensorRT dengan EXPLICIT_BATCH (batch eksplisit)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))

# Membuat parser untuk mem-parsing model ONNX
parser = trt.OnnxParser(network, logger)
parser.parse_from_file(str(onnx_path))  # Mem-parsing model ONNX dari file

# Menyiapkan konfigurasi builder dan profil optimasi
config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # Mengatur batas memori untuk ruang kerja

# Membuat profil optimasi untuk input
profile = builder.create_optimization_profile()
profile.set_shape("input", (32, 1, 28, 28), (32, 1, 28, 28), (32, 1, 28, 28))
config.add_optimization_profile(profile)

# Menyusun dan menyerialisasi jaringan, kemudian menyimpan engine ke disk
serialized_engine = builder.build_serialized_network(network, config)
with open(str(trt_path), 'wb') as f:
    f.write(serialized_engine)

# Membebaskan sumber daya yang digunakan oleh builder dan network
del builder
del network


## Jalankan Inferensi dan Periksa Akurasi

Terakhir, jalankan inferensi lalu bandingkan akurasi model mesin TensorRT dengan model ONNX pada set data pengujian.

Untuk menjalankan pengujian model ONNX, muat model dan uji integritas model model lalu ulangi Data Loader yang diberikan. Untuk setiap batch, konversikan data input ke array NumPy dan masukkan ke dalam sesi ONNX Runtime. Sekali, diperoleh keluaran yang diubah kembali menjadi tensor PyTorch. Kemudian, hitung akumulasi kerugian kemungkinan log negatif
dan jumlah prediksi yang benar untuk mengukur keakuratan model.

Untuk menguji model tensorRT, pertama-tama, muat mesin serial dari disk, dan inisialisasi runtime TensorRT. Kemudian, deserialisasi mesin dan buat konteks eksekusi dibuat. Selanjutnya, alokasikan memori untuk data input dan output pada GPU, atur binding untuk eksekusi TensorRT, dan buat aliran CUDA untuk mengelola transfer data asinkron antara CPU dan GPU. Kemudian, Ulangi Data Loader yang diberikan dan untuk setiap batch, konversikan data input ke array NumPy dan transfer ke GPU, sebelum mengeksekusi model secara asinkron, lalu transfer prediksi kembali ke CPU. Jalankan sinkronisasi untuk memastikan koordinasi yang tepat antar thread. Selanjutnya, bentuk ulang keluaran dan konversikan ke tensor PyTorch untuk menghitung akumulasi kerugian kemungkinan log negatif dan jumlah prediksi yang benar untuk mengukur keakuratan model. Terakhir, kosongkan memori dan sumber daya CUDA

In [5]:
# Fungsi untuk mengonversi tensor PyTorch menjadi array NumPy
def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

def test_onnx(model_name, data_loader):
    onnx_model = onnx.load(model_name)  # Memuat model ONNX dari file
    onnx.checker.check_model(onnx_model)  # Memeriksa validitas model ONNX
    ort_session = onnxruntime.InferenceSession(model_name)  # Membuat sesi inferensi dengan ONNX Runtime
    test_loss = 0
    correct = 0
    for data, target in data_loader:
        ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(data)}  # Mengonversi input menjadi NumPy
        output = ort_session.run(None, ort_inputs)[0]  # Melakukan inferensi pada data
        output = torch.from_numpy(output)  # Mengonversi hasil kembali ke tensor PyTorch
        if target.shape[0] == 32:  # Memeriksa apakah ukuran batch adalah 32, beberapa batch terakhir mungkin lebih kecil
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # Menjumlahkan loss batch
            pred = output.argmax(dim=1, keepdim=True)  # Mendapatkan indeks dengan probabilitas tertinggi
            correct += pred.eq(target.view_as(pred)).sum().item()  # Menjumlahkan prediksi yang benar
    test_loss /= len(data_loader.dataset)  # Rata-rata loss
    return 100. * correct / len(data_loader.dataset)  # Menghitung akurasi
def test_tensorrt(model_name, data_loader):
    with open(model_name, "rb") as f:
        serialized_engine = f.read()  # Membaca engine serialized dari file
    runtime = trt.Runtime(logger)  # Membuat runtime untuk TensorRT
    engine = runtime.deserialize_cuda_engine(serialized_engine)  # Membaca serialized engine ke dalam format TensorRT
    context = engine.create_execution_context()  # Membuat konteks eksekusi untuk model
    input_size = trt.volume(engine.get_binding_shape(0))  # Mendapatkan ukuran input tensor
    output_size = trt.volume(engine.get_binding_shape(1))  # Mendapatkan ukuran output tensor
    # Mengalokasikan memori untuk input dan output pada GPU
    d_input = cuda.mem_alloc(input_size * 4)  # Mengalokasikan memori untuk input (asumsi float32)
    d_output = cuda.mem_alloc(output_size * 4)  # Mengalokasikan memori untuk output
    bindings=[int(d_input), int(d_output)]  # Bindings untuk input dan output tensor
    stream = cuda.Stream()  # Membuat stream untuk asinkronisasi
    h_output = np.empty(output_size, dtype=np.float32)  # Membuat array kosong untuk output pada CPU
    test_loss = 0
    correct = 0
    for data, target in data_loader:
        h_input = data.numpy().astype(np.float32)  # Mengonversi data menjadi array NumPy dan float32
        # Transfer data input ke GPU
        cuda.memcpy_htod_async(d_input, h_input, stream)
        # Menjalankan inferensi dengan TensorRT
        context.execute_async_v2(bindings, stream.handle, None)
        # Transfer hasil inferensi kembali ke CPU
        cuda.memcpy_dtoh_async(h_output, d_output, stream)
        # Sinkronisasi thread GPU
        stream.synchronize()
        output = h_output.reshape(context.get_tensor_shape('output'))  # Menyesuaikan bentuk output
        output = torch.from_numpy(output)  # Mengonversi hasil kembali menjadi tensor PyTorch
        if target.shape[0] == 32:  # Memeriksa jika batch terakhir lebih kecil dari 32
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # Menjumlahkan loss
            pred = output.argmax(dim=1, keepdim=True)  # Mendapatkan prediksi kelas
            correct += pred.eq(target.view_as(pred)).sum().item()  # Menjumlahkan prediksi yang benar
    test_loss /= len(data_loader.dataset)  # Rata-rata loss
    del context  # Menghapus konteks TensorRT
    del engine  # Menghapus engine TensorRT
    cuda.Context.pop()  # Menghapus konteks CUDA
    return 100. * correct / len(data_loader.dataset)  # Menghitung akurasi

# Menghitung akurasi untuk model ONNX
acc = test_onnx(onnx_path, test_loader)
print(f"Accuracy of the onnx model is {acc}%")

# Menghitung akurasi untuk model TensorRT
trtr_acc = test_tensorrt(trt_path, test_loader)
print(f"Accuracy of the tensorrt model is {trtr_acc}%")


Accuracy of the onnx model is 96.2%


<ipython-input-5-e7f70e092b44>:27: DeprecationWarning: Use get_tensor_shape instead.
  input_size = trt.volume(engine.get_binding_shape(0))
<ipython-input-5-e7f70e092b44>:28: DeprecationWarning: Use get_tensor_shape instead.
  output_size = trt.volume(engine.get_binding_shape(1))


Accuracy of the tensorrt model is 96.2%
